In [1]:
# ======= #
# Imports #                   
# ======= #
import pandas as pd
import numpy as np
import re
import json
import joblib
from pathlib import Path
from collections import Counter
from itertools import combinations

from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from scipy.spatial.distance import cosine, jensenshannon
from collections import defaultdict

from joblib import dump
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
print('Imports ok.')


from su_utils import deserialize_tuple, normalize_to_distribution, primary_from_labels, eval_distribution


Imports ok.


ModuleNotFoundError: No module named 'su_utils'

## 2. Configuration and Data Loading

In [ ]:
#DATA_DIR = Path("/kaggle/input/prepreprocessed-for-bert")
DATA_DIR = Path("/kaggle/input/lis070-su-admin-data")
MODEL_DIR = Path("/kaggle/input/kbbert-swe/kbbert_swe")

train_ml_export = pd.read_csv(DATA_DIR / "train_ml_export.csv")
val_df   = pd.read_csv(DATA_DIR / "val_ml_export.csv")
label_list = joblib.load(DATA_DIR / "uo_label_list.joblib")
#deserialize back to tuples
val_df["labels_uo"] = val_df["labels_uo"].apply(deserialize_tuple)
val_df["labels_pct"] = val_df["labels_pct"].apply(deserialize_tuple)


print ("dirs ok")




dirs ok


---
## 3. Prepare Data for Training

In [ ]:
# Identify label columns
label_cols = [c for c in train_ml_export.columns if c.startswith("y_")]
num_labels = len(label_cols)

train_texts = train_ml_export["text"].astype(str).tolist()
val_texts   = val_df["text"].astype(str).tolist()

Y_train = train_ml_export[label_cols].values.astype("float32")
Y_val   = val_df[label_cols].values.astype("float32")
print("Done")
print(f"Number of labels: {num_labels}")
print(f"Label columns: {label_cols}")

Done


In [ ]:
#Build HuggingFace datasets
from datasets import Dataset

train_hf = Dataset.from_dict({"text": train_texts, "labels": list(Y_train)})
val_hf   = Dataset.from_dict({"text": val_texts, "labels": list(Y_val)})
print(f"Train dataset: {train_hf}")
print(f"Val dataset: {val_hf}")

Done


---
## 4. Load Model and Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)

def tokenize_batch(batch):
    enc = tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=512,
    )
    enc["labels"] = batch["labels"]
    return enc

train_tokenized = train_hf.map(tokenize_batch, batched=True)
val_tokenized   = val_hf.map(tokenize_batch, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    local_files_only=True,
    ignore_mismatched_sizes=True,  # FIX 1
)
print(f"Model loaded: {model.config.architectures}")
print(f"Output labels: {num_labels}")

Map:   0%|          | 0/7865 [00:00<?, ? examples/s]

Map:   0%|          | 0/1905 [00:00<?, ? examples/s]

2025-12-10 22:48:12.822231: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765406892.987303      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765406893.033778      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/kbbert-swe/kbbert_swe and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Done


---
## 5. Define Metrics (move to util file)

In [ ]:
#move to util file
from sklearn.metrics import f1_score, accuracy_score, hamming_loss
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # labels will come in as shape (batch, num_labels)
    preds = (1 / (1 + np.exp(-logits)) >= 0.5).astype(int)

    # subset accuracy = exact match of all labels
    subset_acc = accuracy_score(labels, preds)
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    hamming = hamming_loss(labels, preds)

    return {
        "subset_accuracy": subset_acc,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "hamming_loss": hamming,
    }
print("Metrics function defined")

---
## 6. Training

In [ ]:
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./kbbert_uo_multilabel",
    report_to="none",  # Disisable wandb
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()
metrics = trainer.evaluate()
metrics

Epoch,Training Loss,Validation Loss,Subset Accuracy,Micro F1,Macro F1,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.074400,0.074564,0.885564,0.921751,0.881125,0.018583,34.021700,55.994000,1.764000
2,0.041500,0.056693,0.899738,0.931224,0.900843,0.016430,34.518900,55.187000,1.738000
3,0.020900,0.053967,0.907612,0.935470,0.904603,0.015433,33.815500,56.335000,1.774000


{'eval_loss': 0.05396690219640732,
 'eval_subset_accuracy': 0.9076115485564304,
 'eval_micro_f1': 0.9354697102721685,
 'eval_macro_f1': 0.9046030552413425,
 'eval_hamming_loss': 0.015433070866141733,
 'eval_runtime': 34.2307,
 'eval_samples_per_second': 55.652,
 'eval_steps_per_second': 1.753,
 'epoch': 3.0}

In [ ]:
# Final evaluation
binary_metrics = trainer.evaluate()

print("\n" + "="*60)
print("BINARY CLASSIFICATION RESULTS")
print("="*60)
print(f"Subset Accuracy: {binary_metrics['eval_subset_accuracy']:.4f}")
print(f"Micro F1:        {binary_metrics['eval_micro_f1']:.4f}")
print(f"Macro F1:        {binary_metrics['eval_macro_f1']:.4f}")
print(f"Hamming Loss:    {binary_metrics['eval_hamming_loss']:.4f}")

---
## 7. Generate Predictions and Distributions

Convert binary sigmoid outputs to normalized distributions for comparison with the direct distributional approach.

In [ ]:
# Get raw logits from the model on the validation set
pred_out = trainer.predict(val_tokenized)
logits = pred_out.predictions              # shape: (n_val, num_labels)

# Convert logits to probabilities via sigmoid
Y_prob = 1 / (1 + np.exp(-logits))        # same as torch.sigmoid but in NumPy

# Binary predictions (threshold 0.5)
Y_pred_binary = (Y_prob >= 0.5).astype(int)
# Normalize to distribution (sum to 100%)
pred_dists = normalize_to_distribution(Y_prob, scale=100)
pred_dists.shape
print(f"Predictions shape: {Y_prob.shape}")
print(f"Distribution sums (first 5): {pred_dists[:5].sum(axis=1)}")

(1905, 10)

---
## 8. Build Gold Distributions

In [ ]:
# UO code to name mapping
UO_NAMES = {
    2434: "HU",  # Humaniora
    2436: "JU",  # Juridik
    2438: "LU",  # Lärarutbildning  
    2439: "ME",  # Medicin
    2441: "NA",  # Naturvetenskap
    2442: "SA",  # Samhällsvetenskap
    2444: "TE",  # Teknik
    2445: "VÅ",  # Vård
    2447: "ÖV",  # Övrigt
    2451: "VU"   # Verksamhetsförlagd utbildning
}
def get_uo_name(code):
    """Get human-readable UO name."""
    return UO_NAMES.get(code, f"UO {code}")

# Build mapping
uo_to_idx = {code: i for i, code in enumerate(label_list)}
num_labels = len(label_list)

def make_gold_distribution(row):
    """
    Convert (labels_uo, labels_pct) tuples into a distribution vector.
    """
    dist = np.zeros(num_labels, dtype=float)
    uos = row["labels_uo"]
    pcts = row["labels_pct"]
    
    if not isinstance(uos, (list, tuple)) or len(uos) == 0:
        return dist
    
    if not isinstance(pcts, (list, tuple)) or len(pcts) != len(uos):
        equal_pct = 100.0 / len(uos)
        pcts = [equal_pct] * len(uos)
    
    for uo, pct in zip(uos, pcts):
        if uo in uo_to_idx:
            dist[uo_to_idx[uo]] = pct
    
    return dist

# Build gold distribution array
gold_dist = np.array([make_gold_distribution(row) for _, row in val_df.iterrows()])

print(f"Gold distribution shape: {gold_dist.shape}")
print(f"Gold distribution sums (first 5): {gold_dist[:5].sum(axis=1)}")

Gold distribution shape: (1905, 10)
Sample gold dist (first 3 rows):
[[  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0. 100.   0.   0.   0.   0.   0.]]


---
## 9. Evaluate Distributional Metrics

In [ ]:
#Run evaluation
dist_metrics = eval_distribution(gold_dist, pred_dists, scale=100)
per_sample_mae = dist_metrics["per_sample_mae"]

print("\n" + "="*60)
print("DISTRIBUTIONAL METRICS (Post-hoc Normalized)")
print("="*60)
print(f"MAE (percentage points): {dist_metrics['mae_pct']:.2f}")
print(f"Top-1 Accuracy:          {dist_metrics['top1_accuracy']:.4f} ({dist_metrics['top1_accuracy']*100:.1f}%)")
print(f"Mean Cosine Similarity:  {dist_metrics['mean_cosine_sim']:.4f}")
print(f"Mean JSD:                {dist_metrics['mean_js_divergence']:.4f}")

In [ ]:
# Per-label analysis
gold_primary = np.argmax(gold_dist, axis=1)
pred_primary = np.argmax(pred_dists, axis=1)

print("\n" + "="*60)
print("PER-LABEL ANALYSIS")
print("="*60)

# Per-label MAE
per_label_mae = np.abs(gold_dist - pred_dists).mean(axis=0)
label_mae_df = pd.DataFrame({
    "uo_code": label_list,
    "uo_name": [get_uo_name(c) for c in label_list],
    "mae_pct": per_label_mae
}).sort_values("mae_pct", ascending=False)

print("\nPer-Label MAE (percentage points):")
print(label_mae_df.to_string(index=False))

# Per-label primary accuracy
print("\nPer-Label Primary Accuracy:")
for i, code in enumerate(label_list):
    mask = gold_primary == i
    if mask.sum() > 0:
        acc = (pred_primary[mask] == i).mean()
        print(f"  {get_uo_name(code):30s}: {acc:.1%} ({mask.sum()} cases)")


DISTRIBUTIONAL EVALUATION METRICS
Mean Absolute Error (percentage points): 2.79
Mean Cosine Similarity:                  0.9382
Mean Jensen-Shannon Divergence:          0.1915
Top-1 Accuracy (primary label match):    0.8525


In [ ]:
def get_worst_predictions(val_df, gold_dist, pred_dist, per_sample_mae, n=15):
    """
    Get detailed info on worst predictions.
    """
    worst_idx = np.argsort(per_sample_mae)[-n:][::-1]
    
    rows = []
    for idx in worst_idx:
        row = val_df.iloc[idx]
        
        gold_primary_idx = np.argmax(gold_dist[idx])
        pred_primary_idx = np.argmax(pred_dist[idx])
        gold_primary_code = label_list[gold_primary_idx]
        pred_primary_code = label_list[pred_primary_idx]
        
        text = row.get("text", "")
        text_preview = text[:300] + "..." if len(text) > 300 else text
        
        rows.append({
            "id": row["id"],
            "mae": per_sample_mae[idx],
            "gold_labels": dict(zip(row["labels_uo"], row["labels_pct"])) if row["labels_uo"] else {},
            "gold_primary": f"{gold_primary_code} ({get_uo_name(gold_primary_code)})",
            "pred_primary": f"{pred_primary_code} ({get_uo_name(pred_primary_code)})",
            "pred_primary_pct": f"{pred_dist[idx][pred_primary_idx]:.1f}%",
            "correct_primary": gold_primary_code == pred_primary_code,
            "text_preview": text_preview,
        })
    
    return pd.DataFrame(rows)

worst_df = get_worst_predictions(val_df, gold_dist, pred_dists, per_sample_mae, n=15)

print("\n" + "="*60)
print("WORST PREDICTIONS BY MAE")
print("="*60)
for _, row in worst_df.iterrows():
    print(f"\n{'─'*60}")
    print(f"Course ID: {row['id']} | MAE: {row['mae']:.2f}")
    print(f"Gold: {row['gold_labels']} → {row['gold_primary']}")
    print(f"Pred: {row['pred_primary']} ({row['pred_primary_pct']})")
    print(f"Primary correct: {'✓' if row['correct_primary'] else '✗'}")
    print(f"Text: {row['text_preview']}")


WORST PREDICTIONS (highest MAE)

Course ID: 31197
  Gold:      {2447: 100.0}
  Gold dist: [  0.   0.   0.   0.   0.   0.   0.   0. 100.   0.]
  Pred dist: [96.5  0.3  0.4  0.3  0.6  0.4  0.3  0.3  0.5  0.3]
  MAE:       19.91

Course ID: 14747
  Gold:      {2438: 100.0}
  Gold dist: [  0.   0. 100.   0.   0.   0.   0.   0.   0.   0.]
  Pred dist: [96.8  0.3  0.5  0.2  0.4  0.7  0.3  0.2  0.4  0.3]
  MAE:       19.90

Course ID: 14747
  Gold:      {2438: 100.0}
  Gold dist: [  0.   0. 100.   0.   0.   0.   0.   0.   0.   0.]
  Pred dist: [96.8  0.3  0.5  0.2  0.5  0.6  0.3  0.2  0.4  0.3]
  MAE:       19.90

Course ID: 49441
  Gold:      {2436: 100.0}
  Gold dist: [  0. 100.   0.   0.   0.   0.   0.   0.   0.   0.]
  Pred dist: [25.9  0.6  0.4  0.3  0.2 71.4  0.2  0.3  0.4  0.2]
  MAE:       19.88

Course ID: 43983
  Gold:      {2434: 100.0}
  Gold dist: [100.   0.   0.   0.   0.   0.   0.   0.   0.   0.]
  Pred dist: [ 0.6  0.5  0.4 45.6  0.6 49.8  1.1  0.4  0.5  0.5]
  MAE:       19.

---
## 11. Export for Expert Review

In [ ]:
def create_expert_review_csv(val_df, gold_dist, pred_dist, per_sample_mae, output_path, n_cases=50):
    """
    Create CSV for expert review with stratified selection:
    - Worst predictions by MAE
    - High uncertainty cases (flat predicted distribution)
    - Random sample for baseline
    """
    n_samples = len(val_df)
    
    # Selection indices
    worst_idx = np.argsort(per_sample_mae)[-n_cases//2:][::-1]
    
    pred_entropy = -np.sum(pred_dist/100 * np.log(pred_dist/100 + 1e-10), axis=1)
    uncertain_idx = np.argsort(pred_entropy)[-n_cases//4:][::-1]
    
    np.random.seed(42)
    random_idx = np.random.choice(n_samples, size=n_cases//4, replace=False)
    
    all_idx = list(dict.fromkeys(list(worst_idx) + list(uncertain_idx) + list(random_idx)))[:n_cases]
    
    rows = []
    for idx in all_idx:
        row = val_df.iloc[idx]
        gold_uos = row["labels_uo"]
        gold_pcts = row["labels_pct"]
        
        gold_primary_idx = np.argmax(gold_dist[idx])
        pred_primary_idx = np.argmax(pred_dist[idx])
        gold_primary_code = label_list[gold_primary_idx]
        pred_primary_code = label_list[pred_primary_idx]
        
        if idx in worst_idx:
            selection_reason = "high_mae"
        elif idx in uncertain_idx:
            selection_reason = "high_uncertainty"
        else:
            selection_reason = "random"
        
        record = {
            "id": row["id"],
            "text": row.get("text", ""),
            "selection_reason": selection_reason,
            "mae": round(per_sample_mae[idx], 2),
            "pred_entropy": round(pred_entropy[idx], 4),
            "primary_uo": gold_primary_code,
            "primary_uo_name": get_uo_name(gold_primary_code),
            "labels_uo": str(list(gold_uos)) if gold_uos else "[]",
            "labels_pct": str(list(gold_pcts)) if gold_pcts else "[]",
            "pred_primary_uo": pred_primary_code,
            "pred_primary_uo_name": get_uo_name(pred_primary_code),
            "pred_primary_pct": round(pred_dist[idx][pred_primary_idx], 1),
            "primary_match": gold_primary_code == pred_primary_code,
        }
        
        for i, code in enumerate(label_list):
            record[f"pred_{code}"] = round(pred_dist[idx][i], 2)
            record[f"gold_{code}"] = round(gold_dist[idx][i], 2)
        
        rows.append(record)
    
    df_export = pd.DataFrame(rows).sort_values("mae", ascending=False)
    df_export.to_csv(output_path, index=False)
    
    return df_export

expert_df = create_expert_review_csv(
    val_df, gold_dist, pred_dist, per_sample_mae,
    output_path=OUTPUT_DIR / "bert_binary_expert_review.csv",
    n_cases=50
)

print(f"\nExported {len(expert_df)} cases for expert review")
print(f"  - High MAE: {(expert_df['selection_reason'] == 'high_mae').sum()}")
print(f"  - High uncertainty: {(expert_df['selection_reason'] == 'high_uncertainty').sum()}")
print(f"  - Random: {(expert_df['selection_reason'] == 'random').sum()}")
print(f"  - Primary match rate: {expert_df['primary_match'].mean():.1%}")


LABEL CONFUSION ANALYSIS
(When gold is X, model predicts Y instead)
 gold_code gold_name  pred_code pred_name  confusion_pct  n_cases
      2444        TE       2442        SA      94.594595       70
      2445        VÅ       2442        SA      85.185185       23
      2439        ME       2442        SA      84.507042       60
      2445        VÅ       2438        LU      14.814815        4
      2438        LU       2442        SA      12.328767        9


---
## 12. Summary and Model Export

In [ ]:
# Primary label distribution comparison
print("\n" + "="*60)
print("PRIMARY LABEL DISTRIBUTION")
print("="*60)
print("\nGold vs Predicted primary labels:")
for i, code in enumerate(label_list):
    gold_count = (gold_primary == i).sum()
    pred_count = (pred_primary == i).sum()
    diff = pred_count - gold_count
    print(f"  {get_uo_name(code):30s}: Gold={gold_count:4d}, Pred={pred_count:4d}, Diff={diff:+4d}")

In [ ]:
# Save model and predictions
import torch

# Save model
torch.save(model.state_dict(), "bert_binary_model.pt")

# Save predictions
results_df = val_df[["id", "text", "labels_uo", "labels_pct"]].copy()
results_df["mae"] = per_sample_mae
for i, code in enumerate(label_list):
    results_df[f"pred_{code}"] = pred_dist[:, i]
    results_df[f"gold_{code}"] = gold_dist[:, i]
    results_df[f"prob_{code}"] = Y_prob[:, i]

results_df["gold_primary"] = [label_list[i] for i in gold_primary]
results_df["pred_primary"] = [label_list[i] for i in pred_primary]
results_df["primary_match"] = results_df["gold_primary"] == results_df["pred_primary"]

results_df.to_csv("bert_binary_predictions.csv", index=False)
print(f"✓ Predictions saved to {'bert_binary_predictions.csv'}")



OVERALL SUMMARY

Primary label distribution (Gold vs Predicted):
  HU                       : Gold= 677, Pred= 671, Diff=  -6
  JU                       : Gold=  77, Pred=  76, Diff=  -1
  LU                       : Gold=  73, Pred=  87, Diff= +14
  ME                       : Gold=  71, Pred=  10, Diff= -61
  NA                       : Gold= 321, Pred= 316, Diff=  -5
  SA                       : Gold= 439, Pred= 588, Diff=+149
  TE                       : Gold=  74, Pred=   0, Diff= -74
  VÅ                       : Gold=  27, Pred=   0, Diff= -27
  ÖV                       : Gold= 123, Pred= 135, Diff= +12
  VU                       : Gold=  23, Pred=  22, Diff=  -1

Per-label primary accuracy:
  HU                       : 96.5% (677 cases)
  JU                       : 93.5% (77 cases)
  LU                       : 82.2% (73 cases)
  ME                       : 14.1% (71 cases)
  NA                       : 91.9% (321 cases)
  SA                       : 90.9% (439 cases)
  TE            

In [ ]:
# Final summary
print("\n" + "="*60)
print("TRAINING COMPLETE - FINAL SUMMARY")
print("="*60)

print("\n Binary Classification Metrics:")
print(f"   Subset Accuracy: {binary_metrics['eval_subset_accuracy']:.4f}")
print(f"   Micro F1:        {binary_metrics['eval_micro_f1']:.4f}")
print(f"   Macro F1:        {binary_metrics['eval_macro_f1']:.4f}")
print(f"   Hamming Loss:    {binary_metrics['eval_hamming_loss']:.4f}")

print("\n Distributional Metrics (post-hoc normalized):")
print(f"   MAE:             {dist_metrics['mae_pct']:.2f} pp")
print(f"   Top-1 Accuracy:  {dist_metrics['top1_accuracy']*100:.1f}%")
print(f"   Cosine Sim:      {dist_metrics['mean_cosine_sim']:.4f}")
print(f"   JSD:             {dist_metrics['mean_js_divergence']:.4f}")

print("\n Exported files:")
print(f"   - bert_binary_model.pt")
print(f"   - bert_binary_predictions.csv")
print(f"   - bert_binary_expert_review.csv")

In [25]:
# =============================================================================
# SAVE PREDICTIONS FOR BOOTSTRAP TESTING
# =============================================================================

# Binary predictions (thresholded at 0.5)
Y_pred_bert_binary = (Y_prob >= 0.5).astype(int)

# Save for bootstraping:
np.save("/kaggle/working/bert_binary_Y_pred.npy", Y_pred_bert_binary)  # For multi-label metrics
np.save("/kaggle/working/bert_binary_Y_prob.npy", Y_prob)              # Raw sigmoid probs
np.save("/kaggle/working/bert_binary_pred_dist.npy", pred_dist)        # Normalized to 100

print(f"✓ Saved Y_pred_bert_binary: {Y_pred_bert_binary.shape}")
print(f"✓ Saved Y_prob (sigmoid): {Y_prob.shape}")
print(f"✓ Saved pred_dist (normalized): {pred_dist.shape}")

✓ Saved Y_pred_bert_binary: (1905, 10)
✓ Saved Y_prob (sigmoid): (1905, 10)
✓ Saved pred_dist (normalized): (1905, 10)
